# Demo XAI — Dekoding Metafora Sampul Majalah Tempo

**Multi-Task Learning CLIP–IndoBERT dengan Explainable AI**

Tanggal: 29 September 2026

---

## Apa isi notebook ini

Notebook ini menjalankan seluruh alur dari gambar sampul sampai penjelasan XAI:

1. Memuat dataset 740 sampul yang sudah dianotasi dan diadjudikasi
2. Melatih head Multi-Task Learning di atas fitur CLIP + IndoBERT
3. Memprediksi tiga task sekaligus
4. Menghasilkan **Grad-CAM** (penjelasan visual) dan **atribusi kata** (penjelasan tekstual)
5. Menyusun **penjelasan naratif** yang setiap bagiannya dapat dilacak asalnya
6. Memverifikasi apakah model "melihat" bagian gambar yang benar

---

## PERINGATAN PENTING — baca sebelum menilai hasil

**Ini prototipe untuk verifikasi alur, bukan hasil akhir penelitian.**

| Hal | Kondisi sekarang |
|---|---|
| Jumlah data | 740 sampul (dataset final) |
| Macro-F1 Task 2 (domain), fusion MTL | **0,320** — naik dari 0,174 pada pilot 189 sampul, masih perlu ditingkatkan |
| Uji leave-one-out 7 sampul | 7 dari 21 prediksi benar (3 task × 7 sampul) |
| Zona Grad-CAM dominan (25 sampul) | 22/25 masih di zona kepala sampul (teks/logo), lihat Bagian 13 |
| Grad-CAM | bukan Grad-CAM CNN standar (lihat Bagian 7) |
| Atribusi kata | occlusion, bukan LIME penuh (lihat Bagian 7) |

Tujuan notebook ini adalah membuktikan **alur XAI berjalan dan dapat diverifikasi**,
bukan menunjukkan performa model. Angka yang muncul adalah hasil model nyata —
tidak ada yang diilustrasikan atau dikarang.


---

## Bagian 1 — Persiapan

Jalankan sel di bawah untuk memasang pustaka yang dibutuhkan.

**Bila memakai Google Colab:** jalankan sel ini, tunggu sampai selesai
(sekitar 2–4 menit), lalu lanjut. Tidak perlu GPU.

**Bila memakai komputer sendiri:** pastikan Python 3.10 atau 3.11.
Python 3.12+ dan 3.14 belum tentu didukung PyTorch.


In [ ]:
# Pasang pustaka. Di Colab cukup sekali per sesi.
import sys, subprocess

PAKET = ["torch", "torchvision", "transformers", "open_clip_torch",
         "scikit-learn", "pandas", "pillow", "matplotlib"]

def sudah_ada(nama):
    peta = {"open_clip_torch": "open_clip", "scikit-learn": "sklearn",
            "pillow": "PIL"}
    try:
        __import__(peta.get(nama, nama))
        return True
    except ImportError:
        return False

kurang = [p for p in PAKET if not sudah_ada(p)]
if kurang:
    print("Memasang:", ", ".join(kurang))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *kurang], check=True)
    print("Selesai.")
else:
    print("Semua pustaka sudah tersedia.")


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image

torch.manual_seed(42)
np.random.seed(42)

print("torch:", torch.__version__)
print("perangkat: CPU (encoder dibekukan, GPU tidak diperlukan)")


### Menentukan lokasi berkas

Notebook ini mencari berkas di folder `03_data` dan `05_gambar` yang ada
di dalam paket yang sama.

**Bila memakai Colab:** unggah dulu seluruh folder paket ini ke Google Drive,
lalu sesuaikan `AKAR` di bawah. Contoh:
`AKAR = Path('/content/drive/MyDrive/DEMO_XAI_LENGKAP_2026-09-09')`


In [ ]:
# Sesuaikan bila perlu. Default: folder induk notebook ini.
AKAR = Path.cwd()
if not (AKAR / "03_data").exists() and (AKAR.parent / "03_data").exists():
    AKAR = AKAR.parent

DATA_CSV = AKAR / "03_data" / "DATASET_740_TASK2_DOMAIN.csv"
FITUR_NPZ = AKAR / "03_data" / "fitur_clip_indobert_740.npz"
DIR_GAMBAR = AKAR / "05_gambar"

for nama, p in [("dataset", DATA_CSV), ("fitur", FITUR_NPZ), ("gambar", DIR_GAMBAR)]:
    print(f"{nama:9} {'OK    ' if p.exists() else 'HILANG'} {p}")

if not DATA_CSV.exists():
    raise SystemExit("Berkas tidak ditemukan. Sesuaikan variabel AKAR di atas.")


---

## Bagian 2 — Memuat dataset

Dataset berisi 740 sampul Majalah Tempo yang sudah dianotasi dua anotator
dan diadjudikasi pihak ketiga.

**Tiga task yang diprediksi:**

| Task | Isi | Jumlah kelas |
|---|---|---|
| Task 1 | Isu utama sampul | 10 |
| Task 2 | Domain sumber metafora | 7 |
| Task 3 | Tipe entitas sasaran | 8 |

**Catatan soal Task 2.** Sebelumnya Task 2 berbentuk biner (ada/tidak ada
metafora). Namun 91,5% sampul Tempo bermetafora, sehingga model cukup menjawab
"ada" terus dan sudah benar 92% — recall kelas non-metafora hanya 0,062.
Karena itu Task 2 digeser menjadi klasifikasi **domain sumber** metafora:
bukan lagi "ada metafora atau tidak", melainkan "metaforanya memakai apa".


In [ ]:
df = pd.read_csv(DATA_CSV, encoding="utf-8-sig")
z = np.load(FITUR_NPZ)
X = np.hstack([z["img"], z["txt"]]).astype(np.float32)
DIM_IMG = z["img"].shape[1]

print(f"Sampul          : {len(df)}")
print(f"Fitur CLIP      : {z['img'].shape[1]} dimensi (visual)")
print(f"Fitur IndoBERT  : {z['txt'].shape[1]} dimensi (teks)")
print(f"Fitur gabungan  : {X.shape[1]} dimensi")
print()
print("Status kelayakan sampul (eligibility):")
print(df["eligibility"].value_counts().to_string())


In [ ]:
TASKS = {
    "task1_issue": "Task 1 — Isu Utama",
    "task2_domain_sumber": "Task 2 — Domain Sumber Metafora",
    "task3_primary_entity_type": "Task 3 — Tipe Entitas Sasaran",
}

peta, y, n_kelas = {}, {}, {}
for t in TASKS:
    kelas = sorted(df[t].astype(str).unique())
    peta[t] = kelas
    y[t] = df[t].astype(str).map({c: i for i, c in enumerate(kelas)}).to_numpy()
    n_kelas[t] = len(kelas)

for t, nama in TASKS.items():
    print(f"\n{nama} — {n_kelas[t]} kelas")
    print(df[t].value_counts().to_string())


### Ketimpangan kelas — hal yang harus dipahami

Perhatikan keluaran di atas. Beberapa kelas hanya punya puluhan sampul,
jauh lebih sedikit dibanding kelas terbesar (lihat Bagian 5 dan 6 untuk
Macro-F1 dan recall per kelas).

Ini bukan kesalahan teknis, melainkan konsekuensi distribusi topik asli pada
740 sampul Majalah Tempo yang dianotasi — beberapa domain metafora memang
jarang dipakai dibanding domain lain.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, sumbu = plt.subplots(1, 3, figsize=(16, 4))
for ax, (t, nama) in zip(sumbu, TASKS.items()):
    hitung = df[t].value_counts().sort_values()
    warna = ["#b91c1c" if v < 15 else "#172554" for v in hitung.values]
    ax.barh(range(len(hitung)), hitung.values, color=warna)
    ax.set_yticks(range(len(hitung)))
    ax.set_yticklabels([s[:24] for s in hitung.index], fontsize=8)
    ax.set_title(nama, fontsize=10)
    ax.set_xlabel("jumlah sampul", fontsize=9)
plt.suptitle("Distribusi kelas — merah = di bawah 15 sampul (terlalu sedikit)",
             fontsize=11)
plt.tight_layout()
plt.savefig("distribusi_kelas.png", dpi=110, bbox_inches="tight")
plt.close()
print("Grafik disimpan: distribusi_kelas.png")
from IPython.display import Image as TampilGambar, display
display(TampilGambar("distribusi_kelas.png"))


---

## Bagian 3 — Arsitektur model

Mengikuti rancangan pada proposal:

```
Sampul Tempo
    |
    +-- gambar  --> CLIP ViT-B/32   --> 512 dimensi  (DIBEKUKAN)
    |                                        |
    |                                        +--> digabung --> 1280 dimensi
    |                                        |
    +-- teks    --> IndoBERT-base-p1 --> 768 dimensi  (DIBEKUKAN)
                    (headline [SEP] teks pendukung)
                                                 |
                                        shared layer (256 unit)
                                                 |
                        +------------------------+------------------------+
                        |                        |                        |
                   Task 1 head             Task 2 head             Task 3 head
                   (10 kelas)               (7 kelas)               (8 kelas)
```

**Mengapa encoder dibekukan.** Dengan 740 sampul, melatih ulang ratusan juta
parameter CLIP dan IndoBERT tetap berisiko overfitting. Yang dilatih
hanya shared layer dan tiga head — sekitar 330 ribu parameter. Fine-tuning penuh
baru masuk akal setelah dataset jauh lebih besar, dan tetap perlu diuji bertahap.


In [ ]:
class KepalaMTL(nn.Module):
    """Shared layer + tiga task head. Encoder CLIP/IndoBERT tidak dilatih."""

    def __init__(self, dim_masuk, n_kelas, tersembunyi=256, p_drop=0.4):
        super().__init__()
        self.bersama = nn.Sequential(
            nn.Linear(dim_masuk, tersembunyi),
            nn.ReLU(),
            nn.Dropout(p_drop),
        )
        self.kepala = nn.ModuleDict(
            {t: nn.Linear(tersembunyi, n) for t, n in n_kelas.items()}
        )

    def forward(self, x):
        h = self.bersama(x)
        return {t: k(h) for t, k in self.kepala.items()}


contoh = KepalaMTL(X.shape[1], n_kelas)
jumlah = sum(p.numel() for p in contoh.parameters())
print(f"Parameter yang dilatih : {jumlah:,}")
print(f"Parameter CLIP+IndoBERT: ~197.000.000 (dibekukan, tidak dilatih)")


---

## Bagian 4 — Melatih model

Fungsi di bawah melatih head MTL. Parameter `kecualikan` dipakai untuk
**leave-one-out**: satu sampul dikeluarkan dari data latih agar prediksinya
jujur — model benar-benar belum pernah melihat sampul itu.

**Class weighting aktif.** Kelas yang jarang diberi bobot loss lebih besar.
Perlu dicatat: pada eksperimen sebelumnya, class weighting saja **tidak cukup**
mengatasi ketimpangan 10,8:1 pada Task 2 versi biner.


In [ ]:
def latih(kecualikan=None, epoch=120, lr=1e-3, wd=1e-4, tampilkan=False):
    """Latih head MTL. kecualikan = indeks sampul yang di-hold-out."""
    torch.manual_seed(42)
    idx = np.array([i for i in range(len(df)) if i != kecualikan])

    # bobot kelas dihitung dari data latih saja, bukan seluruh dataset
    bobot = {}
    for t in TASKS:
        cnt = np.bincount(y[t][idx], minlength=n_kelas[t]).astype(np.float32)
        w = np.where(cnt > 0, len(idx) / (np.maximum(cnt, 1) * (cnt > 0).sum()), 0.0)
        bobot[t] = torch.tensor(w, dtype=torch.float32)

    model = KepalaMTL(X.shape[1], n_kelas)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    rugi = {t: nn.CrossEntropyLoss(weight=bobot[t]) for t in TASKS}

    Xt = torch.tensor(X[idx])
    yt = {t: torch.tensor(y[t][idx]) for t in TASKS}

    model.train()
    for e in range(epoch):
        opt.zero_grad()
        keluar = model(Xt)
        total = sum(rugi[t](keluar[t], yt[t]) for t in TASKS) / len(TASKS)
        total.backward()
        opt.step()
        if tampilkan and (e + 1) % 30 == 0:
            print(f"  epoch {e+1:>3}  loss {total.item():.4f}")
    model.eval()
    return model


print("Melatih pada seluruh 740 sampul:")
model_penuh = latih(tampilkan=True)
print("Selesai.")


---

## Bagian 5 — Evaluasi dengan 5-fold cross-validation

Dengan 740 sampul, satu pembagian train/test tunggal tetap kurang stabil untuk
dipercaya sendirian. Karena itu dipakai Stratified 5-Fold: setiap sampul mendapat
kesempatan menjadi data uji tepat satu kali.

**Metrik utama adalah Macro-F1, bukan accuracy.** Pada data timpang, accuracy
menyesatkan — model yang selalu menjawab kelas terbanyak bisa terlihat bagus
padahal tidak belajar apa pun.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from collections import Counter

def evaluasi(fitur, n_lipat=5):
    """Stratified K-Fold. Stratifikasi memakai Task 1 sebagai acuan."""
    strat = y["task1_issue"]
    cnt = Counter(strat)
    aman = np.array([c if cnt[c] >= n_lipat else -1 for c in strat])
    skf = StratifiedKFold(n_splits=n_lipat, shuffle=True, random_state=42)

    tebak = {t: np.zeros(len(strat), dtype=int) for t in TASKS}
    for i_latih, i_uji in skf.split(fitur, aman):
        bobot = {}
        for t in TASKS:
            c = np.bincount(y[t][i_latih], minlength=n_kelas[t]).astype(np.float32)
            w = np.where(c > 0, len(i_latih) / (np.maximum(c, 1) * (c > 0).sum()), 0.0)
            bobot[t] = torch.tensor(w, dtype=torch.float32)

        torch.manual_seed(42)
        m = KepalaMTL(fitur.shape[1], n_kelas)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
        rugi = {t: nn.CrossEntropyLoss(weight=bobot[t]) for t in TASKS}
        Xt = torch.tensor(fitur[i_latih])
        yt = {t: torch.tensor(y[t][i_latih]) for t in TASKS}
        m.train()
        for _ in range(120):
            opt.zero_grad()
            o = m(Xt)
            (sum(rugi[t](o[t], yt[t]) for t in TASKS) / len(TASKS)).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            o = m(torch.tensor(fitur[i_uji]))
        for t in TASKS:
            tebak[t][i_uji] = o[t].argmax(dim=1).numpy()

    hasil = {}
    for t in TASKS:
        hasil[t] = {
            "accuracy": accuracy_score(y[t], tebak[t]),
            "macro_f1": f1_score(y[t], tebak[t], average="macro", zero_division=0),
        }
    return hasil, tebak


print("Menjalankan 5-fold CV untuk empat varian model.")
print("Perlu beberapa menit.\n")

varian = {
    "image-only (CLIP)": X[:, :DIM_IMG],
    "text-only (IndoBERT)": X[:, DIM_IMG:],
    "fusion MTL (CLIP+IndoBERT)": X,
}

# baseline: selalu menebak kelas terbanyak
hasil_semua = {"majority baseline": {}}
for t in TASKS:
    mayoritas = Counter(y[t]).most_common(1)[0][0]
    tebak_maj = np.full(len(y[t]), mayoritas)
    hasil_semua["majority baseline"][t] = {
        "accuracy": accuracy_score(y[t], tebak_maj),
        "macro_f1": f1_score(y[t], tebak_maj, average="macro", zero_division=0),
    }

tebakan_fusion = None
for nama, fitur in varian.items():
    print(f"  {nama} ...")
    h, tb = evaluasi(fitur)
    hasil_semua[nama] = h
    if nama.startswith("fusion"):
        tebakan_fusion = tb
print("\nSelesai.")


In [ ]:
baris = []
for nama, h in hasil_semua.items():
    baris.append({
        "model": nama,
        "T1 Macro-F1": round(h["task1_issue"]["macro_f1"], 3),
        "T2 Macro-F1": round(h["task2_domain_sumber"]["macro_f1"], 3),
        "T3 Macro-F1": round(h["task3_primary_entity_type"]["macro_f1"], 3),
    })
tabel = pd.DataFrame(baris).set_index("model")
print("MACRO-F1 — metrik utama untuk data timpang\n")
print(tabel.to_string())
print()
print("Bandingkan tiap model dengan 'majority baseline'.")
print("Model yang tidak melampaui baseline berarti belum mempelajari apa pun.")


### Recall per kelas — melihat kelas mana yang gagal

Macro-F1 hanya memberi satu angka rata-rata. Untuk mengetahui **kelas mana**
yang benar-benar tidak dipelajari, perlu dilihat recall per kelas.


In [ ]:
t = "task2_domain_sumber"
p, r, f, dukung = precision_recall_fscore_support(
    y[t], tebakan_fusion[t], average=None, zero_division=0,
    labels=list(range(n_kelas[t]))
)
rinci = pd.DataFrame({
    "kelas": peta[t],
    "jumlah": dukung,
    "recall": np.round(r, 3),
    "f1": np.round(f, 3),
}).sort_values("jumlah", ascending=False)

print(f"{TASKS[t]} — recall per kelas (fusion MTL)\n")
print(rinci.to_string(index=False))

gagal = rinci[rinci["recall"] == 0]
if len(gagal):
    print(f"\n{len(gagal)} kelas dengan recall 0 — tidak pernah diprediksi sama sekali:")
    for _, x in gagal.iterrows():
        print(f"   {x['kelas']} (hanya {x['jumlah']} sampul)")
    print("\nIni konsekuensi jumlah sampel, bukan kesalahan arsitektur.")


---

## Bagian 6 — Memuat encoder CLIP dan IndoBERT

Sampai bagian ini kita memakai fitur yang sudah dihitung sebelumnya. Untuk
Grad-CAM, encoder aslinya harus dimuat karena kita perlu mengalirkan gradien
kembali ke lapisan visual.

Unduhan pertama sekitar 1,5 GB dan butuh beberapa menit. Setelah itu tersimpan
di cache.


In [ ]:
import open_clip
from transformers import AutoModel, AutoTokenizer

MODEL_CLIP = "ViT-B-32"
PRETRAIN_CLIP = "laion2b_s34b_b79k"
MODEL_BERT = "indobenchmark/indobert-base-p1"

print("Memuat CLIP ...")
clip_model, _, praproses = open_clip.create_model_and_transforms(
    MODEL_CLIP, pretrained=PRETRAIN_CLIP
)
clip_model.eval()

print("Memuat IndoBERT ...")
tok = AutoTokenizer.from_pretrained(MODEL_BERT)
bert = AutoModel.from_pretrained(MODEL_BERT)
bert.eval()

print("\nSiap.")


---

## Bagian 7 — Grad-CAM pada CLIP ViT

### Dua penyesuaian yang WAJIB disebut di laporan

**Grad-CAM standar tidak berfungsi pada CLIP ViT.** Ini bukan dugaan, tetapi
hasil penelusuran gradien: `encode_image` CLIP hanya memakai token CLS untuk
menghasilkan embedding akhir. Akibatnya, pada blok transformer **terakhir**
gradien seluruh 49 token patch bernilai **nol persis**. Rumus Grad-CAM CNN
mengalikan aktivasi patch dengan bobot gradien — bila gradiennya nol, seluruh
peta panas menjadi kosong.

Dua penyesuaian yang dilakukan:

1. **Hook diambil pada blok kedua dari belakang** (`resblocks[-2]`), bukan
   terakhir. Pada blok itu informasi patch masih mengalir ke CLS melalui
   self-attention, sehingga gradien patch tidak nol.

2. **Besaran pengaruh per patch dihitung sebagai |aktivasi × gradien|**
   (Gradient × Activation), bukan rumus Grad-CAM CNN.

Selain itu, peta panas **diregangkan kontrasnya berbasis peringkat persentil**
sebelum diwarnai. Urutan patch tidak berubah, hanya keterlihatannya, karena peta
mentah ViT memusat pada sedikit patch dengan ekor mendekati nol.

**Kesimpulan: ini bukan Grad-CAM CNN yang identik, dan tidak boleh disebut
demikian di laporan.** Hal ini mengonfirmasi risiko metodologis yang sudah
tercatat sejak awal: Grad-CAM tidak berlaku langsung pada arsitektur Transformer.


In [ ]:
def grad_cam(jalur_gambar, fitur_teks, task, model):
    """Peta panas 7x7 dari blok transformer kedua-dari-belakang CLIP ViT-B/32.

    Mengembalikan (cam, indeks_kelas) atau (None, indeks) bila gradien nol.
    """
    gambar = praproses(Image.open(jalur_gambar).convert("RGB")).unsqueeze(0)
    simpan = {}

    def maju(_m, _i, o):
        simpan["akt"] = o

    def mundur(_m, _gi, go):
        simpan["grad"] = go[0]

    # blok -2, bukan -1. Alasannya dijelaskan di sel markdown di atas.
    blok = clip_model.visual.transformer.resblocks[-2]
    h1 = blok.register_forward_hook(maju)
    h2 = blok.register_full_backward_hook(mundur)

    try:
        v = clip_model.encode_image(gambar)
        v = v / v.norm(dim=-1, keepdim=True)
        x = torch.cat([v[0], torch.tensor(fitur_teks)]).reshape(1, -1)
        logit = model(x)[task][0]
        k = int(logit.argmax())
        clip_model.zero_grad(set_to_none=True)
        model.zero_grad(set_to_none=True)
        logit[k].backward()
    finally:
        h1.remove()
        h2.remove()

    a, g = simpan.get("akt"), simpan.get("grad")
    if a is None or g is None:
        return None, k

    a, g = a.detach(), g.detach()
    if a.dim() == 3:
        if a.shape[0] == 1:
            a, g = a[0], g[0]
        elif a.shape[1] == 1:
            a, g = a[:, 0, :], g[:, 0, :]
        else:
            return None, k

    a, g = a[1:], g[1:]          # buang token CLS
    if float(g.abs().sum()) == 0.0:
        return None, k

    cam = (a * g).sum(dim=-1).abs()
    n = cam.shape[0]
    sisi = int(round(n ** 0.5))
    if sisi * sisi != n:
        return None, k

    cam = cam.reshape(sisi, sisi).float().numpy()
    rentang = float(cam.max() - cam.min())
    cam = (cam - cam.min()) / rentang if rentang > 1e-12 else np.zeros_like(cam)
    return cam, k


print("Fungsi grad_cam siap.")


In [ ]:
def tempel_peta_panas(jalur_gambar, cam, jalur_keluar, alpha=0.62):
    """Tempelkan peta panas ke gambar asli.

    Nilai diregangkan memakai peringkat persentil agar terlihat; urutan patch
    tidak berubah.
    """
    img = Image.open(jalur_gambar).convert("RGB")
    w, h = img.size

    datar = cam.flatten()
    urut = datar.argsort().argsort().astype(np.float32)
    peringkat = urut / (len(datar) - 1) if len(datar) > 1 else np.zeros_like(datar)
    camr = peringkat.reshape(cam.shape) ** 1.6

    panas = Image.fromarray((camr * 255).astype(np.uint8)).resize((w, h), Image.BICUBIC)
    p = np.asarray(panas).astype(np.float32) / 255.0

    r = np.clip(2.0 * p - 0.6, 0, 1)
    g = np.clip(1.6 * (1.0 - np.abs(p - 0.55) * 2.0), 0, 1)
    b = np.clip(1.15 - 2.0 * p, 0, 1)
    warna = np.stack([r, g, b], axis=-1)

    dasar = np.asarray(img).astype(np.float32) / 255.0
    kuat = (p[..., None] ** 0.85) * alpha
    hasil = dasar * (1 - kuat) + warna * kuat
    Image.fromarray((np.clip(hasil, 0, 1) * 255).astype(np.uint8)).save(jalur_keluar)
    return jalur_keluar


def zona_dominan(cam):
    """Bagi peta panas menjadi tiga zona dan cari yang paling berpengaruh."""
    n = cam.shape[0]
    atas = max(1, n // 3)
    bawah = n - max(1, n // 3)
    zona = {
        "kepala sampul (logo dan judul kecil)": float(cam[:atas].mean()),
        "bagian tengah (umumnya ilustrasi)": float(cam[atas:bawah].mean()),
        "kaki sampul (headline besar dan barcode)": float(cam[bawah:].mean()),
    }
    urut = sorted(zona.items(), key=lambda kv: -kv[1])
    return urut[0][0], urut[0][1], zona


print("Fungsi visualisasi siap.")


---

## Bagian 8 — Atribusi kata (occlusion)

Metode: setiap kata pada headline dihapus satu per satu, lalu diukur penurunan
probabilitas kelas terpilih. Kata yang penghapusannya menurunkan probabilitas
paling banyak adalah kata yang paling menopang keputusan.

**Ini bukan LIME penuh.** LIME memakai model surrogate dan weighted sampling di
sekitar instance. Metode di sini lebih sederhana, tetapi deterministik (hasilnya
selalu sama) dan cukup untuk memperlihatkan kata mana yang berpengaruh.
Bila laporan menuntut LIME sebenarnya, itu pekerjaan terpisah.


In [ ]:
def fitur_teks_dari(headline, pendukung=""):
    """Hitung embedding IndoBERT untuk 'headline [SEP] pendukung'."""
    with torch.no_grad():
        teks = f"{headline} [SEP] {pendukung}".strip()
        enc = tok(teks, return_tensors="pt", truncation=True,
                  max_length=128, padding="max_length")
        h = bert(**enc).last_hidden_state[:, 0, :]   # token CLS
        h = h / h.norm(dim=-1, keepdim=True)
    return h.squeeze(0).numpy().astype(np.float32)


def atribusi_kata(fitur_gambar, headline, pendukung, task, model):
    """Kontribusi tiap kata pada headline terhadap kelas terpilih."""
    def prob(teks_headline):
        ft = fitur_teks_dari(teks_headline, pendukung)
        x = np.concatenate([fitur_gambar, ft]).reshape(1, -1).astype(np.float32)
        with torch.no_grad():
            o = model(torch.tensor(x))[task][0]
        return torch.softmax(o, dim=0).numpy()

    dasar = prob(headline)
    k = int(dasar.argmax())
    kata = headline.split()
    skor = []
    for i in range(len(kata)):
        sisa = " ".join(kata[:i] + kata[i + 1:])
        p = prob(sisa) if sisa.strip() else dasar * 0
        skor.append({"kata": kata[i], "kontribusi": float(dasar[k] - p[k])})
    return skor, k, float(dasar[k])


print("Fungsi atribusi kata siap.")


---

## Bagian 9 — Penjelasan naratif

Bagian ini menjawab permintaan agar output tidak hanya berupa skor, tetapi juga
paragraf penjelasan.

### Prinsip yang dipegang: setiap teks harus dapat dilacak asalnya

Ada dua cara membuat penjelasan naratif, dan pilihan ini menentukan sah-tidaknya
untuk laporan penelitian:

| | Cara A (dipakai) | Cara B (tidak dipakai) |
|---|---|---|
| Sumber teks | dirangkai dari prediksi model, catatan anotator, dan definisi Codebook | dihasilkan model bahasa |
| Dapat dilacak | ya, setiap bagian diberi penanda asal | tidak |
| Risiko mengarang | tidak ada | ada — dapat menulis penjelasan meyakinkan untuk prediksi yang salah |
| Termasuk XAI | ya | tidak — teks generate bukan penjelasan mekanisme model |

**Cara B ditolak** karena tiga alasan. Pertama, XAI menjelaskan *mengapa model
mengambil keputusan*; teks yang digenerate adalah teks baru, bukan penjelasan
mekanisme. Kedua, model dapat menulis penjelasan yang mulus untuk prediksi yang
salah, sehingga menutupi kesalahan alih-alih memperlihatkannya. Ketiga, bila
ditanya asal paragrafnya, jawabannya "dari model bahasa lain", dan itu
melemahkan klaim XAI.

Karena itu, `DEFINISI_DOMAIN` di bawah **ditulis manusia** berdasarkan pembacaan
173 catatan pemetaan makna. Teksnya tetap, tidak dihasilkan saat notebook
dijalankan.


In [ ]:
DEFINISI_DOMAIN = {
    "HEWAN_TUBUH": {
        "definisi": "Metafora yang memakai hewan, bagian tubuh, atau deformasi "
                    "tubuh sebagai sumber kiasan.",
        "pola": "Pada korpus ini umumnya untuk menggambarkan sifat rakus, licik, "
                "kebohongan, atau dehumanisasi. Contoh: tikus untuk pihak yang "
                "menikmati korupsi; hidung panjang untuk janji yang diingkari.",
    },
    "PERMAINAN_KONTES": {
        "definisi": "Metafora yang memakai permainan, pertandingan, sulap, atau "
                    "panggung pertunjukan sebagai sumber kiasan.",
        "pola": "Umumnya untuk menggambarkan politik sebagai sesuatu yang diatur, "
                "dipertaruhkan, atau dipertontonkan.",
    },
    "MESIN_INFRASTRUKTUR": {
        "definisi": "Metafora yang memakai mesin, kendaraan, alat kerja, atau "
                    "bangunan sebagai sumber kiasan.",
        "pola": "Umumnya untuk menggambarkan sistem yang rusak, dikendalikan, "
                "atau dibongkar-pasang.",
    },
    "KONSUMSI_TRANSAKSI": {
        "definisi": "Metafora yang memakai makanan, jamuan, uang, atau kegiatan "
                    "jual-beli sebagai sumber kiasan.",
        "pola": "Umumnya untuk menggambarkan pembagian keuntungan atau selera "
                "pribadi penguasa.",
    },
    "BENCANA_JERAT": {
        "definisi": "Metafora yang memakai bencana alam, api, banjir, atau alat "
                    "pengekang seperti jeruji dan jerat sebagai sumber kiasan.",
        "pola": "Umumnya untuk menggambarkan ancaman yang datang, kekacauan, "
                "atau keterkungkungan.",
    },
    "LAINNYA": {
        "definisi": "Metafora yang sumbernya tidak masuk kelima domain di atas.",
        "pola": "Kelompok ini masih perlu ditinjau manusia; terdapat 63 sampul, "
                "antara lain kiasan ritual pengorbanan dan kematian simbolis.",
    },
    "NON_METAFORA": {
        "definisi": "Sampul yang tidak memuat metafora visual-verbal.",
        "pola": "Umumnya edisi khusus biografi/sejarah dan foto jurnalistik. "
                "83 dari 740 sampul.",
    },
}
print(f"Definisi untuk {len(DEFINISI_DOMAIN)} domain — ditulis manusia, bukan digenerate.")


In [ ]:
def susun_penjelasan(prediksi, task, kata, catatan, cam):
    """Rangkai penjelasan. Setiap blok diberi penanda asal.

    asal: 'model' | 'anotator' | 'codebook' | 'gabungan'
    Fungsi ini hanya menyusun; tidak ada teks yang dihasilkan model bahasa.
    """
    blok = []
    p = prediksi[task]

    # 1. prediksi
    teks = (f"Model memprediksi {p['prediksi']} dengan keyakinan "
            f"{p['probabilitas']*100:.1f}%.")
    if len(p["semua"]) > 1:
        k2 = p["semua"][1]
        teks += f" Pilihan kedua adalah {k2['kelas']} ({k2['prob']*100:.1f}%)."
    if p["probabilitas"] < 0.40:
        teks += (" Keyakinan ini rendah, sehingga prediksinya belum dapat "
                 "dianggap meyakinkan.")
    blok.append(("model", "Prediksi model", teks))

    # 2. kata dominan
    if kata:
        urut = sorted(kata, key=lambda x: -x["kontribusi"])
        atas = urut[0]
        teks = (f'Kata paling menopang keputusan adalah "{atas["kata"]}" '
                f'(kontribusi +{atas["kontribusi"]:.3f}).')
        lain = [x for x in urut[1:] if x["kontribusi"] > 0
                and x["kata"].lower() != atas["kata"].lower()]
        if lain:
            teks += f' Disusul "{lain[0]["kata"]}" (+{lain[0]["kontribusi"]:.3f}).'
        negatif = [x for x in urut if x["kontribusi"] < 0]
        if negatif:
            teks += (f' Sebaliknya, "{negatif[-1]["kata"]}" justru melemahkan '
                     f'keputusan ({negatif[-1]["kontribusi"]:.3f}).')
        blok.append(("model", "Kata dominan", teks))

    # 3. zona visual
    if cam is not None:
        nama, nilai, semua = zona_dominan(cam)
        rinci = ", ".join(f"{k.split('(')[0].strip()} {v:.3f}" for k, v in semua.items())
        teks = (f"Area gambar yang paling memengaruhi keputusan adalah {nama}, "
                f"intensitas rata-rata {nilai:.3f}. Rincian: {rinci}.")
        if "kepala" in nama or "kaki" in nama:
            teks += (" Perlu dicatat: area ini didominasi teks dan logo, bukan "
                     "ilustrasi. Bila prediksinya benar, kemungkinan model bertumpu "
                     "pada teks tercetak, bukan pada gambar metaforanya.")
        blok.append(("model", "Area visual dominan", teks))

    # 4. pemetaan makna (anotator) atau definisi domain (codebook)
    dom = prediksi.get("task2_domain_sumber", {}).get("prediksi")
    if catatan and catatan.get("pemetaan"):
        teks = (f'Anotator mencatat sumber metaforanya adalah '
                f'"{catatan.get("sumber","")}", dengan pemetaan makna: '
                f'{catatan["pemetaan"]}')
        if not teks.rstrip().endswith("."):
            teks += "."
        blok.append(("anotator", "Pemetaan makna metafora", teks))
    elif dom in DEFINISI_DOMAIN:
        d = DEFINISI_DOMAIN[dom]
        blok.append(("codebook", f"Definisi domain {dom}",
                     f"{d['definisi']} {d['pola']}"))

    # 5. verifikasi
    if catatan and catatan.get("sumber") and cam is not None:
        nama, _, _ = zona_dominan(cam)
        if "tengah" in nama:
            teks = ("Area panas berada di bagian tengah, tempat ilustrasi metafora "
                    "biasanya berada. Ini sejalan dengan catatan anotator, sehingga "
                    "keputusan model tampak diambil dari elemen yang tepat.")
        else:
            teks = ("Area panas TIDAK berada di bagian tengah, padahal anotator "
                    "mencatat sumber metaforanya ada pada elemen visual. Artinya "
                    "prediksi model bisa benar tetapi dengan alasan yang salah. "
                    "Temuan seperti ini justru penting dilaporkan.")
        blok.append(("gabungan", "Verifikasi peta panas", teks))

    return blok


print("Fungsi susun_penjelasan siap.")


---

## Bagian 10 — Fungsi analisis lengkap

Fungsi `analisis_sampul` di bawah menyatukan semuanya: prediksi tiga task,
Grad-CAM, atribusi kata, dan penjelasan naratif.

**Leave-one-out.** Sampul yang dianalisis dikeluarkan dari data latih, sehingga
model benar-benar belum pernah melihatnya. Ini penting agar hasilnya jujur —
bukan hasil menghafal.


In [ ]:
from IPython.display import display, Image as TampilGambar, HTML

def analisis_sampul(sample_id, task="task2_domain_sumber", tampilkan=True):
    """Analisis satu sampul dari dataset dengan leave-one-out."""
    baris = df[df["sample_id"] == sample_id]
    if baris.empty:
        raise ValueError(f"sample_id tidak ditemukan: {sample_id}")
    i = int(baris.index[0])
    r = baris.iloc[0]

    jalur = DIR_GAMBAR / f"{sample_id}.jpg"
    if not jalur.exists():
        for ext in (".jpeg", ".png", ".webp"):
            alt = DIR_GAMBAR / f"{sample_id}{ext}"
            if alt.exists():
                jalur = alt
                break
    if not jalur.exists():
        raise FileNotFoundError(f"gambar tidak ada: {sample_id}")

    # latih tanpa sampul ini
    model = latih(kecualikan=i)

    fitur_gambar = X[i, :DIM_IMG]
    fitur_teks = X[i, DIM_IMG:]

    with torch.no_grad():
        keluar = model(torch.tensor(X[i].reshape(1, -1)))
    prediksi = {}
    for t in TASKS:
        prob = torch.softmax(keluar[t][0], dim=0).numpy()
        urut = prob.argsort()[::-1]
        prediksi[t] = {
            "nama": TASKS[t],
            "prediksi": peta[t][int(urut[0])],
            "probabilitas": float(prob[urut[0]]),
            "label_asli": str(r[t]),
            "cocok": peta[t][int(urut[0])] == str(r[t]),
            "semua": [{"kelas": peta[t][int(j)], "prob": float(prob[j])} for j in urut],
        }

    cam, _ = grad_cam(jalur, fitur_teks, task, model)
    berkas_panas = None
    if cam is not None:
        berkas_panas = f"peta_panas_{sample_id}.png"
        tempel_peta_panas(jalur, cam, berkas_panas)

    headline = str(r["headline_verified"])
    pendukung = str(r.get("supporting_text_verified", "") or "")
    kata, _, _ = atribusi_kata(fitur_gambar, headline, pendukung, task, model)

    catatan = {
        "sumber": str(r.get("task2_source_description", "") or ""),
        "pemetaan": str(r.get("task2_mapping_note", "") or ""),
    }
    penjelasan = susun_penjelasan(prediksi, task, kata, catatan, cam)

    if tampilkan:
        cocok = sum(1 for t in TASKS if prediksi[t]["cocok"])
        print("=" * 78)
        print(f"{sample_id}   {headline}")
        print("=" * 78)
        print()
        print("PREDIKSI TIGA TASK")
        for t in TASKS:
            p = prediksi[t]
            tanda = "COCOK" if p["cocok"] else "TIDAK COCOK"
            print(f"  {p['nama']}")
            print(f"     model    : {p['prediksi']} ({p['probabilitas']*100:.1f}%)")
            print(f"     anotator : {p['label_asli']}  ->  {tanda}")
        print(f"\n  Skor: {cocok} dari 3 task cocok dengan anotator")
        print()
        if berkas_panas:
            print("PENJELASAN VISUAL — Grad-CAM")
            display(TampilGambar(berkas_panas, width=380))
        print()
        print("PENJELASAN NARATIF")
        print("-" * 78)
        for asal, label, teks in penjelasan:
            print(f"[{asal.upper()}] {label}")
            for potong in [teks[j:j+74] for j in range(0, len(teks), 74)]:
                print(f"   {potong}")
            print()

    return {"prediksi": prediksi, "penjelasan": penjelasan,
            "kata": kata, "cam": cam, "peta_panas": berkas_panas}


print("Fungsi analisis_sampul siap.")


---

## Bagian 11 — Contoh: sampul Pinokio

Sampul **"JANJI TINGGAL JANJI"** memakai metafora bayangan hidung Pinokio untuk
menggambarkan janji yang diingkari. Ini contoh yang paling sering dipakai untuk
menjelaskan konsep dekoding metafora.

Perhatikan blok **[ANOTATOR]** — di situlah penjelasan "Pinokio = kebohongan"
muncul, dan asalnya adalah anotasi manusia, bukan karangan model.


In [ ]:
cari = df[df["headline_verified"].str.contains("JANJI TINGGAL", case=False, na=False)]
if len(cari):
    hasil_pinokio = analisis_sampul(cari.iloc[0]["sample_id"])
else:
    print("Sampul tidak ditemukan; menampilkan contoh pertama.")
    hasil_pinokio = analisis_sampul(df.iloc[0]["sample_id"])


---

## Bagian 12 — Contoh lain: sampul tikus (SKANDAL BNI)

Sampul ini memakai metafora tikus berpesta untuk menggambarkan pihak yang
menikmati hasil korupsi. Bandingkan apakah peta panasnya menyala di area
tikusnya.


In [ ]:
cari = df[df["headline_verified"].str.contains("SKANDAL BNI", case=False, na=False)]
if len(cari):
    hasil_bni = analisis_sampul(cari.iloc[0]["sample_id"])


---

## Bagian 13 — Uji beberapa sampul sekaligus

Menganalisis satu sampul per domain untuk melihat pola secara lebih luas.
Bagian ini memerlukan beberapa menit.


In [ ]:
terpilih = []
sudah = set()
for _, r in df.iterrows():
    d = r["task2_domain_sumber"]
    if d not in sudah:
        terpilih.append(r["sample_id"])
        sudah.add(d)

print(f"Menganalisis {len(terpilih)} sampul (satu per domain).\n")

rekap = []
for sid in terpilih:
    h = analisis_sampul(sid, tampilkan=False)
    cocok = sum(1 for t in TASKS if h["prediksi"][t]["cocok"])
    zona = zona_dominan(h["cam"])[0] if h["cam"] is not None else "-"
    rekap.append({
        "headline": df[df.sample_id == sid].iloc[0]["headline_verified"][:30],
        "domain asli": df[df.sample_id == sid].iloc[0]["task2_domain_sumber"],
        "prediksi": h["prediksi"]["task2_domain_sumber"]["prediksi"],
        "cocok/3": cocok,
        "zona dominan": zona.split("(")[0].strip(),
    })

tabel_rekap = pd.DataFrame(rekap)
print(tabel_rekap.to_string(index=False))
print()
total_cocok = tabel_rekap["cocok/3"].sum()
total_uji = len(tabel_rekap) * 3
print(f"Total: {total_cocok} dari {total_uji} prediksi cocok dengan anotator")


### Analisis zona: apakah model melihat ilustrasi atau teks?

Ini temuan penting. Grid Grad-CAM dibagi tiga zona, lalu dihitung zona mana yang
paling sering dominan.

Bila yang dominan adalah **kepala sampul** (logo dan judul kecil), berarti model
lebih bertumpu pada teks tercetak daripada ilustrasi metaforanya — padahal
justru ilustrasi yang memuat metafora.


In [ ]:
print("Menghitung zona dominan pada 25 sampul pertama.")
print("Perlu beberapa menit.\n")

zona_rekap = {"kepala": 0, "tengah": 0, "kaki": 0}
nilai_zona = {"kepala": [], "tengah": [], "kaki": []}

for sid in df["sample_id"].head(25):
    i = int(df.index[df.sample_id == sid][0])
    jalur = DIR_GAMBAR / f"{sid}.jpg"
    if not jalur.exists():
        for ext in (".jpeg", ".png", ".webp"):
            alt = DIR_GAMBAR / f"{sid}{ext}"
            if alt.exists():
                jalur = alt
                break
    if not jalur.exists():
        continue
    m = latih(kecualikan=i)
    cam, _ = grad_cam(jalur, X[i, DIM_IMG:], "task2_domain_sumber", m)
    if cam is None:
        continue
    nama, _, semua = zona_dominan(cam)
    kunci = "kepala" if "kepala" in nama else ("tengah" if "tengah" in nama else "kaki")
    zona_rekap[kunci] += 1
    for k, v in semua.items():
        kk = "kepala" if "kepala" in k else ("tengah" if "tengah" in k else "kaki")
        nilai_zona[kk].append(v)

print("Zona dominan (dari 25 sampul):")
for k, v in zona_rekap.items():
    print(f"   {k:8} {v:>3} sampul")
print()
print("Intensitas rata-rata per zona:")
for k, v in nilai_zona.items():
    if v:
        print(f"   {k:8} {np.mean(v):.3f}")
print()
if zona_rekap["kepala"] > zona_rekap["tengah"]:
    print("TEMUAN: model lebih bertumpu pada zona kepala sampul (teks dan logo)")
    print("daripada bagian tengah (ilustrasi metafora).")
    print()
    print("Kemungkinan penyebab:")
    print("  1. Fitur CLIP dibekukan, dan CLIP peka pada teks tercetak dalam gambar")
    print("  2. Logo TEMPO dan judul kecil posisinya konsisten, menjadi jalan pintas")
    print("  3. Head hanya dilatih pada 740 sampul, belum belajar mengabaikan area konstan")
    print()
    print("Ini bukti bahwa prediksi yang benar belum tentu karena alasan yang benar.")
    print("Justru inilah gunanya XAI: memperlihatkan kelemahan yang tidak terlihat")
    print("dari angka Macro-F1 saja.")


---

## Bagian 14 — Menganalisis gambar sampul sendiri

Untuk gambar di luar dataset, tidak ada label anotator sehingga tidak ada
verifikasi otomatis. Penjelasan makna diambil dari **definisi Codebook**, bukan
dari catatan anotator.


In [ ]:
def analisis_gambar_baru(jalur_gambar, headline, pendukung="",
                         task="task2_domain_sumber"):
    """Analisis gambar sampul di luar dataset.

    Model dilatih pada seluruh 740 sampul karena tidak ada yang perlu di-hold-out.
    """
    jalur_gambar = Path(jalur_gambar)
    if not jalur_gambar.exists():
        raise FileNotFoundError(jalur_gambar)

    model = latih()   # seluruh 740 sampul

    with torch.no_grad():
        gambar = praproses(Image.open(jalur_gambar).convert("RGB")).unsqueeze(0)
        v = clip_model.encode_image(gambar)
        v = v / v.norm(dim=-1, keepdim=True)
        fitur_gambar = v.squeeze(0).numpy().astype(np.float32)
    fitur_teks = fitur_teks_dari(headline, pendukung)
    vektor = np.concatenate([fitur_gambar, fitur_teks]).astype(np.float32)

    with torch.no_grad():
        keluar = model(torch.tensor(vektor.reshape(1, -1)))
    prediksi = {}
    for t in TASKS:
        prob = torch.softmax(keluar[t][0], dim=0).numpy()
        urut = prob.argsort()[::-1]
        prediksi[t] = {
            "nama": TASKS[t],
            "prediksi": peta[t][int(urut[0])],
            "probabilitas": float(prob[urut[0]]),
            "semua": [{"kelas": peta[t][int(j)], "prob": float(prob[j])} for j in urut],
        }

    cam, _ = grad_cam(jalur_gambar, fitur_teks, task, model)
    berkas = None
    if cam is not None:
        berkas = f"peta_panas_baru_{jalur_gambar.stem}.png"
        tempel_peta_panas(jalur_gambar, cam, berkas)

    kata, _, _ = atribusi_kata(fitur_gambar, headline, pendukung, task, model)
    penjelasan = susun_penjelasan(prediksi, task, kata, None, cam)

    print("=" * 78)
    print(headline)
    print("=" * 78)
    print("\nCATATAN: gambar di luar dataset, tidak ada label anotator")
    print("sehingga tidak ada verifikasi otomatis.\n")
    for t in TASKS:
        p = prediksi[t]
        print(f"  {p['nama']}: {p['prediksi']} ({p['probabilitas']*100:.1f}%)")
    print()
    if berkas:
        display(TampilGambar(berkas, width=380))
    print("\nPENJELASAN NARATIF")
    print("-" * 78)
    for asal, label, teks in penjelasan:
        print(f"[{asal.upper()}] {label}")
        for potong in [teks[j:j+74] for j in range(0, len(teks), 74)]:
            print(f"   {potong}")
        print()
    return {"prediksi": prediksi, "penjelasan": penjelasan, "peta_panas": berkas}


print("Fungsi analisis_gambar_baru siap.")
print()
print("Cara memakai — hapus tanda # pada baris di bawah lalu sesuaikan:")
print()
print('# hasil = analisis_gambar_baru(')
print('#     "sampul_saya.jpg",')
print('#     headline="SET BUSET PERAMPASAN ASET",')
print('#     pendukung="Sejumlah politikus partai penguasa ...",')
print('# )')


---

## Bagian 15 — Ringkasan dan batasan

### Yang sudah terbukti berjalan

| Komponen | Status |
|---|---|
| Fitur CLIP + IndoBERT | berjalan |
| Fusi multimodal, tiga task MTL | berjalan |
| Grad-CAM pada CLIP ViT | berjalan, dengan dua penyesuaian (Bagian 7) |
| Atribusi kata | berjalan, metode occlusion |
| Penjelasan naratif | berjalan, setiap bagian dapat dilacak asalnya |
| Verifikasi terhadap anotasi manusia | berjalan |

### Batasan yang harus disebut di laporan

1. **Macro-F1 Task 2 domain masih 0,320** (fusion MTL, dataset final 740
   sampul) — naik dari 0,174 pada pilot 189 sampul, tetapi beberapa kelas
   (mis. KONSUMSI_TRANSAKSI, MESIN_INFRASTRUKTUR) masih punya recall rendah
   (lihat Bagian 5 dan 6).

2. **Grad-CAM bukan Grad-CAM CNN standar.** Hook pada blok kedua-dari-belakang,
   dan besaran dihitung sebagai Gradient × Activation. Lihat Bagian 7.

3. **Atribusi kata bukan LIME penuh.** Metode occlusion tanpa model surrogate.

4. **Model mengklasifikasi, tidak menjelaskan.** Penjelasan makna metafora
   berasal dari anotasi manusia atau definisi Codebook. Model tidak "memahami"
   makna metafora.

5. **Model cenderung bertumpu pada zona kepala sampul**, bukan ilustrasi
   metaforanya — 22 dari 25 sampul yang diuji (Bagian 13). Ini temuan yang
   perlu ditindaklanjuti.

### Yang belum dikerjakan

- Cohen's Kappa untuk batch di luar pilot
- Training final dengan fixed split dan multi-seed
- LIME sebagai metode baku (sudah diuji dan dipakai di `02_web_demo`, belum
  diintegrasikan ke notebook demo ini)
- Uji hipotesis bias zona atas (misalnya dengan memangkas area logo)

### Catatan penutup

Angka pada notebook ini adalah hasil model nyata. Tidak ada yang diilustrasikan,
dan tidak ada paragraf yang dihasilkan model bahasa. Bila suatu bagian belum
dikerjakan, hal itu dinyatakan apa adanya.
